### Расширенные стратегии чанкинга и реранкинга

Базовый чанкинг по абзацам работает для простых текстов, но для технической документации нужны более умные подходы.

**Семантический чанкинг (через эмбеддинги)**

```Установите зависимости:```

In [1]:
pip install langchain_experimental langchain_ollama -q

Note: you may need to restart the kernel to use updated packages.


! **Идея:** разбивать текст не по фиксированному размеру, а по **смене смысла.**

In [2]:
long_document = """
Программа конференции — это план мероприятий, который включает расписание секций, заседаний, лекций, перерывов и других событий. Она может быть доступна на сайтах организаторов или в специальных разделах на ресурсах, связанных с мероприятием.
Например, программа конференции, посвящённой инвазивным видам растений, может включать следующие разделы и расписание:
26 мая (вторник). Регистрация, секция «Законодательство и правоприменительная практика в отношении инвазивных видов» в конференц-зале ФГБУ «ВНИИКР».
27 мая (среда). Регистрация, пленарное заседание в Красном зале Президиума РАН (Москва, Ленинский проспект, 32а).
28 мая (четверг). Регистрация, секции «Защита и карантин растений. Инвазивные виды: биология и контроль», «Борщевик Сосновского: меры борьбы», «Биологическая защита растений», «Фитосанитарная ботаника и гербология» в разных залах ФГБУ «ВНИИКР».
29 мая (пятница). Регистрация, секции «Выявление, мониторинг и технологии защиты от инвазивных видов», «Частные и общие вопросы защиты и карантина растений», «Оценка инвазивного потенциала видов, прогноз их проникновения, оценка ущерба» в разных залах ФГБУ «ВНИИКР».
"""

In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embeddings = OllamaEmbeddings(model="mxbai-embed-large")
splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",  # или "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=45  # порог чувствительности
)

chunks = splitter.split_text(long_document)
print(f"Создано {len(chunks)} семантических чанков")

/opt/anaconda3/envs/LLM_course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/tm/42g46xn57410g1m6qmw0bh540000gn/T/ipykernel_45541/1277948344.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Создано 7 семантических чанков


In [4]:
chunks

['\nПрограмма конференции — это план мероприятий, который включает расписание секций, заседаний, лекций, перерывов и других событий. Она может быть доступна на сайтах организаторов или в специальных разделах на ресурсах, связанных с мероприятием. Например, программа конференции, посвящённой инвазивным видам растений, может включать следующие разделы и расписание:\n26 мая (вторник). Регистрация, секция «Законодательство и правоприменительная практика в отношении инвазивных видов» в конференц-зале ФГБУ «ВНИИКР».',
 '27 мая (среда).',
 'Регистрация, пленарное заседание в Красном зале Президиума РАН (Москва, Ленинский проспект, 32а). 28 мая (четверг).',
 'Регистрация, секции «Защита и карантин растений.',
 'Инвазивные виды: биология и контроль», «Борщевик Сосновского: меры борьбы», «Биологическая защита растений», «Фитосанитарная ботаника и гербология» в разных залах ФГБУ «ВНИИКР». 29 мая (пятница).',
 'Регистрация, секции «Выявление, мониторинг и технологии защиты от инвазивных видов», «Ч

**Как работает:** модель вычисляет эмбеддинги для предложений и ищет «разрывы» в семантическом пространстве. Там, где смысл резко меняется — ставится граница чанка.

* **Плюсы:** сохраняет смысловую целостность.
* **Минусы:** медленнее фиксированного чанкинга, требует эмбеддинг-модели.

### Иерархический чанкинг (для структурированных документов)

Идея: учитывать иерархию заголовков (##, ###, ####) при разбивке

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False
)

chunks = splitter.split_text(long_document)

In [6]:
chunks

['Программа конференции — это план мероприятий, который включает расписание секций, заседаний, лекций, перерывов и других событий. Она может быть доступна на сайтах организаторов или в специальных разделах на ресурсах, связанных с мероприятием.\nНапример, программа конференции, посвящённой инвазивным видам растений, может включать следующие разделы и расписание:',
 '26 мая (вторник). Регистрация, секция «Законодательство и правоприменительная практика в отношении инвазивных видов» в конференц-зале ФГБУ «ВНИИКР».\n27 мая (среда). Регистрация, пленарное заседание в Красном зале Президиума РАН (Москва, Ленинский проспект, 32а).',
 '28 мая (четверг). Регистрация, секции «Защита и карантин растений. Инвазивные виды: биология и контроль», «Борщевик Сосновского: меры борьбы», «Биологическая защита растений», «Фитосанитарная ботаника и гербология» в разных залах ФГБУ «ВНИИКР».',
 '29 мая (пятница). Регистрация, секции «Выявление, мониторинг и технологии защиты от инвазивных видов», «Частные и 

### Практический чеклист: как подбирать chunk_size

| Тип контента |	Рекомендуемый размер |	Overlap |	Стратегия |
| :--- | :--- | :--- | :--- |
| API-документация (Markdown) |	400–600 токенов |	50–100 |	По структуре заголовков|
Логи, сырой текст|	300–400 токенов	|30–50	|Фиксированный + overlap
Юридические документы|	500–800 токенов	|100–150|	Семантический или по абзацам
Диалоги, чаты|	200–300 токенов	|20–30	|По сообщениям/репликам

---
### Reranking: дообучение под домен

Базовые reranker-модели (TinyBERT, MultiBERT) работают хорошо на общих задачах. Но если у вас **специфичный домен** (медицина, юриспруденция, внутренние термины компании), дообучение может дать +5–15% к точности.

**Когда стоит дообучать reranker**

* У вас есть ≥500–1000 размеченных пар (query, relevant_doc).
* Базовая модель систематически ошибается на ваших типах запросов.
* Вы уже измерили baseline и хотите выжать максимум из пайплайна.
* Не дообучайте на старте: сначала добейтесь работы базового RAG.

**Формат данных для дообучения**

```
# JSONL-формат (одна пара на строку)
{"query": "Как откатить миграцию?", "positive": "alembic downgrade -- ревертит последнюю миграцию", "negative": "alembic upgrade -- применяет миграции"}
{"query": "Создание пользователя через API", "positive": "POST /api/v1/users {\"name\": \"...\"}", "negative": "DELETE /api/v1/users/{id} удаляет пользователя"}
```

**Минимальный пример дообучения (на базе SentenceTransformers)**

In [7]:
pip install sentence_transformers datasets 'accelerate>=0.26.0' -q

Note: you may need to restart the kernel to use updated packages.


In [8]:
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader

# 1. Подготовка данных
train_examples = [
    InputExample(texts=["Как откатить миграцию?",
                       "alembic downgrade -- ревертит последнюю миграцию",
                       "alembic upgrade -- применяет миграции"])
    # ... больше примеров
]

# 2. Загрузка базовой модели
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Настройка обучения
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model)

# 4. Запуск (на бесплатном Colab GPU)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    output_path="./reranker-finetuned"
)

/var/folders/tm/42g46xn57410g1m6qmw0bh540000gn/T/ipykernel_45541/2557778875.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, SentenceTransformer, losses
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7583.04it/s]
/opt/anaconda3/envs/LLM_course/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.87it/s]


**Совет:** начинайте с малого — 100–200 примеров, 1–2 эпохи. Часто даже небольшая донастройка даёт заметный прирост на узком домене.

#### Интеграция кастомного reranker в пайплайн

### Полезно знать

**Чеклист: чанкинг + reranking в продакшене**

* Начните с chunk_size=500, overlap=50 и стратегии по заголовкам.
* Добавьте метаданные source, doc_id — для фильтрации и отладки.
* Векторный поиск: берите top-10–20 кандидатов для reranking.
* Фильтруйте по метаданным до reranking (в Qdrant/Chroma).
* Reranking: flashrank + базовая модель на старте, дообучение — позже, если нужно.
* Логируйте score reranking и финальный выбор — для анализа точности.
* Тестируйте на реальных запросах, а не синтетике.

**Примечание**

* **Overlap — это не «лишние токены», а страховка:** он предотвращает потерю контекста на границах чанков.
* **Reranking добавляет ~100–300 мс** на запрос — учитывайте это в SLA.
* **Дообучение reranker — не серебряная пуля:** сначала настройте чанкинг и метаданные, потом измеряйте, стоит ли усложнять.